# 01 — Prepare training data
Mount Drive, verify frozen manifests, stage data, and load/build the full token cache. No model training and no TEST evaluation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, shutil, subprocess
DRIVE_ROOT='/content/drive/MyDrive/evo2_dissertation'
REPO_ROOT='/content/evo2-dissertation'
LOCAL_ROOT='/content/evo2_dissertation_data'
LOCAL_CACHE=f'{LOCAL_ROOT}/caches/tokens'


## Verify manifests before staging
This integrity-only check confirms hashes/counts and the TEST lock; it does not inspect TEST outcomes.

In [ ]:
subprocess.run(['python',f'{REPO_ROOT}/scripts/verify_data_manifest.py','--drive-root',DRIVE_ROOT,'--verify-fasta-hashes'], check=True)


## Stage to local SSD and load/build cache

In [ ]:
cache_manifest=pathlib.Path(DRIVE_ROOT)/'caches/tokens/token_cache_manifest.json'
stage_cmd=['python',f'{REPO_ROOT}/scripts/stage_drive_data.py','--drive-root',DRIVE_ROOT,'--local-root',LOCAL_ROOT]
if not cache_manifest.exists(): stage_cmd.append('--include-fasta')
subprocess.run(stage_cmd, check=True)
local_manifest=pathlib.Path(LOCAL_CACHE)/'token_cache_manifest.json'
if local_manifest.exists():
    subprocess.run(['python',f'{REPO_ROOT}/scripts/build_token_cache.py','--drive-root',LOCAL_ROOT,'--output-dir',LOCAL_CACHE,'--verify-only'], check=True)
else:
    subprocess.run(['python',f'{REPO_ROOT}/scripts/build_token_cache.py','--drive-root',LOCAL_ROOT,'--output-dir',LOCAL_CACHE], check=True)


## Persist newly built cache back to Drive atomically

In [ ]:
if not cache_manifest.exists():
    from evo2_distill.utils.io import atomic_copy
    drive_cache=pathlib.Path(DRIVE_ROOT)/'caches/tokens'
    for file in pathlib.Path(LOCAL_CACHE).glob('*'):
        if file.is_file(): atomic_copy(file, drive_cache/file.name)
print('CACHE READY; TEST ACCESSED: NO')
